# # **AutoGen sample for ramadan workout recommendation**

In this script we'll use AutoGen framework to build an agent for workout recommandation during Ramadan u


## Step 1 : Import Python Packages

In [2]:
import os
import json

import requests
from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import UserMessage
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_core import CancellationToken
from autogen_core.tools import FunctionTool
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console
from typing import Any, Callable, Set, Dict, List, Optional


## Step 2 : Create Client

We can use the Github Model for access to the LLM. 
The model is defined as gpt-4o-mini but we'll try other models to compare result
To run a quick test  we can the prompt " what is ramadan ? "

In [26]:
Client = AzureAIChatCompletionClient(
    model="gpt-4o-mini",
    endpoint="https://models.inference.ai.azure.com",
    credential=AzureKeyCredential(os.environ["GITHUB_TOKEN"]),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)
result = await Client.create([UserMessage(content="What is Ramadan", source="user")])
print(result)

finish_reason='stop' content='Ramadan is the ninth month of the Islamic lunar calendar and is considered one of the holiest months for Muslims. It is observed by Muslims worldwide as a month of fasting, prayer, reflection, and community. During Ramadan, Muslims fast from dawn until sunset, refraining from food, drink, smoking, and marital relations during daylight hours. The fast, known as "sawm," is intended to promote self-discipline, spiritual growth, and empathy for those who are less fortunate.\n\nIn addition to fasting, Muslims are encouraged to increase their prayers and recitation of the Quran, engage in charitable acts, and seek forgiveness for past sins. The month concludes with the celebration of Eid al-Fitr, a festival that marks the end of the fasting period and is a time for communal prayers, feasting, and giving to charity.\n\nThe exact dates of Ramadan vary each year, as the Islamic calendar is based on lunar cycles, resulting in Ramadan shifting approximately 10 to 12 

## Step 3 : Define the functions

We will give the agent access to a tool that is a function with a list of workout type and their availability. 

The scenario will be a coach agent who have an access to a fitness center database for example.

In [44]:
from typing import Any, Callable, Set, Dict, List, Optional

def workout_availability(workout: str) -> tuple[str, str]:
    """
    Get the recommandation of a workout.
    Args:
        workout(str): The name of the workout to check recommandation.
    Returns:
        tuple[str, str]: contains the workout name and the recommandation.
    """
    ramadan_workout = {
        "yoga": "Recommended",
        "cardio": "Recommended",
        "strength training": "Not Recommended",
        "zumba": "Recommended",
        "cycling": "Not Recommended",
        "pilates": "Recommended",
        "kickboxing": "Not Recommended",
        "dance": "Recommended",
        "boxing": "Not Recommended",
        "Core Workout" : "Not Recommended",
    }
      
    if workout in ramadan_workout:
        return workout, ramadan_workout[workout]
    else:
        return workout, "Not recommended"

## Step 4: define the function tool

To have the agent use the function workout_availability as a function_tool we have to define it as one. A description has to be provided here to help the agent identify what that tool is used for in relation to the task the user has requested.


In [45]:
get_workout_availability = FunctionTool(
    workout_availability, description="Search for the availability of a workout.")

# Step 5 : Define the agent

Now that the client and the tool are created, we can proceed to the agent creation. We define the system_message to give the agent instructions to help find the best workouts during ramadan. 

In [48]:
# Step 5: Define the agent

agent = AssistantAgent(
    name="fitness_coach",
    model_client=Client,
    system_message="You are a fitness coach. Help users find the best workouts during Ramadan.",
    tools=[get_workout_availability],
    reflect_on_tool_use=True,
)


## Step 6 : Run the agent

Now we can run the agent with the message asking the best workout plan to develop abs during ramadan


In [50]:
# Run the agent with a message asking for the best workout plan to develop abs during Ramadan
async def assistant_run() -> None:
    response = await agent.on_messages(
    [TextMessage(content="What workout are recommended during ramadan", source="user")],
        cancellation_token=CancellationToken(),
    )

    print(response.inner_messages)
    print(response.chat_message)
# Use asyncio.run(assistant_run()) when running in a script.
await assistant_run()

[]
source='fitness_coach' models_usage=RequestUsage(prompt_tokens=343, completion_tokens=317) content="During Ramadan, it's important to choose workouts that respect your fasting schedule and help you maintain your energy levels. Here are some recommended workouts you can do:\n\n1. **Light Cardio**: Engaging in light to moderate cardio such as walking, gentle jogging, or cycling can help maintain your fitness without exhausting you.\n\n2. **Stretching and Flexibility Exercises**: Incorporating stretching routines (like yoga or Pilates) improves flexibility and relaxation, which can be beneficial during Ramadan.\n\n3. **Low-Intensity Strength Training**: Focus on bodyweight exercises or light weights for strength training. Examples include:\n   - Bodyweight squats\n   - Push-ups (modified if needed)\n   - Lunges\n   - Resistance band exercises\n\n4. **Core Exercises**: As mentioned earlier, gentle core workouts such as planks, dead bugs, and bridges can be beneficial if done lightly.\n\